In [37]:
import numpy as np
import pyvista as pv
from scipy.spatial import KDTree
import sys
sys.path.append("..")
from error_func import error

case_name = "case2"
mesh_name = "coil_box"
solver1 = "moose"
solver2 = "comsol"

sol_moose = pv.read(f"../../output/{case_name}/{case_name}_{solver1}.vtu")
sol_comsol = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")

In [38]:
tree = KDTree(sol_moose.points)
distances, indices = tree.query(sol_comsol.points)

max_dist = np.max(distances)
print(f"Maximum alignment error (distance): {max_dist:.6e}")
if max_dist > 1e-4:
    print("Warning: Large distance detected. Are the geometries identical?")

aligned_grid = sol_comsol.copy()

for array_name in sol_moose.point_data.keys():
        data = sol_moose.point_data[array_name]
        
        reordered_data = data[indices]
        
        aligned_grid.point_data[array_name] = reordered_data
        print(f"Transferred array: {array_name}")

aligned_grid.save(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")

Maximum alignment error (distance): 8.403429e-11
Transferred array: current_density
Transferred array: magnetic_vector_potential_nd
Transferred array: magnetic_flux_density_nd
Transferred array: magnetic_vector_potential
Transferred array: magnetic_flux_density


In [39]:
sol_moose = pv.read(f"../../output/{case_name}/{case_name}_{solver1}_reordered_to_{solver2}.vtu")
# elec_pot_ngsolve = sol_moose["electric_potential"]
mag_flux_ngsolve = sol_moose["magnetic_flux_density"]
mag_vec_ngsolve = sol_moose["magnetic_vector_potential"]

sol_comsol = pv.read(f"../../output/{case_name}/{case_name}_{solver2}.vtu")
# elec_pot_comsol = sol_comsol["electric_potential"]
mag_flux_comsol = sol_comsol["magnetic_flux_density"]
mag_vec_comsol = sol_comsol["magnetic_vector_potential"]

# print(elec_pot_ngsolve.shape)
# print(elec_pot_comsol.shape)

print(mag_flux_ngsolve.shape)
print(mag_flux_comsol.shape)

print(mag_vec_ngsolve.shape)
print(mag_vec_comsol.shape)

(278516, 3)
(278516, 3)
(278516, 3)
(278516, 3)


In [40]:
mesh = sol_comsol.copy()

mesh.point_data.remove("magnetic_vector_potential")
mesh.point_data.remove("magnetic_flux_density")

In [41]:
# print(f"Electric potential errors between {solver1} and {solver2}:")

# mesh = error(sol=elec_pot_ngsolve, sol_ref=elec_pot_comsol, 
#              eps = 1e-6, mesh=mesh, tag="scalar", save_tag="V")

In [42]:
print(f"Magnetic flux density errors between {solver1} and {solver2}:")

mesh = error(sol=mag_flux_ngsolve, sol_ref=mag_flux_comsol,
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="B")

# print(mag_flux_ngsolve[1100, :])
# print(mag_flux_comsol[1100, :])

Magnetic flux density errors between moose and comsol:

  * Max. absolute error in x direction  : 8.400e+04.
  * Avg. absolute error in x direction  : 6.252e+03.

  * Max. relative error in x direction : 1.593e+08 %.
  * Avg. relative error in x direction : 1.142e+03 %.

  * Max. absolute error in y direction  : 7.663e+04.
  * Avg. absolute error in y direction  : 4.245e+03.

  * Max. relative error in y direction : 3.487e+06 %.
  * Avg. relative error in y direction : 3.179e+02 %.

  * Max. absolute error in z direction  : 1.084e+05.
  * Avg. absolute error in z direction  : 9.740e+03.

  * Max. relative error in z direction : 3.304e+06 %.
  * Avg. relative error in z direction : 3.284e+02 %.


In [43]:
print(f"Magnetic vector potential errors between {solver1} and {solver2}:")

mesh = error(sol=mag_vec_ngsolve, sol_ref=mag_vec_comsol, 
             eps = 1e-6, mesh=mesh, tag="vector", save_tag="A")

# print(mag_vec_ngsolve[1100, :])
# print(mag_vec_comsol[1100, :])

Magnetic vector potential errors between moose and comsol:

  * Max. absolute error in x direction  : 6.053e+13.
  * Avg. absolute error in x direction  : 6.428e+11.

  * Max. relative error in x direction : 5.507e+18 %.
  * Avg. relative error in x direction : 2.730e+13 %.

  * Max. absolute error in y direction  : 1.172e+14.
  * Avg. absolute error in y direction  : 1.252e+12.

  * Max. relative error in y direction : 5.744e+16 %.
  * Avg. relative error in y direction : 9.814e+11 %.

  * Max. absolute error in z direction  : 1.316e+14.
  * Avg. absolute error in z direction  : 4.601e+11.

  * Max. relative error in z direction : 4.777e+17 %.
  * Avg. relative error in z direction : 1.025e+13 %.


In [44]:
mesh.save(f"../../output/{case_name}/{case_name}_error_moose_comsol.vtu")